# 08. 자료 찾아 답하기 (RAG)

LLM 은 모르는 것도 그럴듯하게 지어낸다. 그래서 **① 질문과 가까운 자료 조각을 찾아서 ② 그 조각을 보고 답하게** 한다.

1. 글을 조각으로 나눈다
2. 조각마다 **임베딩**(뜻을 담은 숫자 384개)을 만든다 — multilingual-e5-small
3. 질문도 임베딩하고, **코사인 유사도**가 큰 조각을 고른다
4. 고른 조각 + 질문을 LLM(VLM 의 글 모드)에 넣어 답을 받는다

## 1. 임베딩 모델 — 토크나이저도 OpenVINO 모델이다

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

import openvino_tokenizers                       # 토크나이저 모델을 여는 확장 — import 해야 연산이 등록된다
tcore = ov.Core()                                # 확장은 import 뒤에 만든 Core 에 붙는다

E = MODELS / "embed" / "e5-small-int8"
tok = tcore.compile_model(E / "openvino_tokenizer.xml", "CPU")
emb = tcore.compile_model(E / "openvino_model.xml", "CPU")
print("임베딩 모델 입력:", [i.any_name for i in emb.inputs])

def embed(texts, kind="passage"):
    """글 목록 → [개수, 384] (길이 1로 맞춤). e5 는 앞에 'passage: ' / 'query: ' 를 붙여야 한다."""
    t = tok([f"{kind}: {s}" for s in texts])
    feed = {"input_ids": t["input_ids"], "attention_mask": t["attention_mask"]}
    if "token_type_ids" in {i.any_name for i in emb.inputs}:
        feed["token_type_ids"] = np.zeros_like(t["input_ids"])
    out = emb(feed)[emb.output(0)]                    # [개수, 토큰 수, 384]
    m = t["attention_mask"][..., None].astype(np.float32)
    v = (out * m).sum(1) / m.sum(1)                    # 진짜 토큰만 평균 (mean pooling)
    return v / np.linalg.norm(v, axis=1, keepdims=True)

v = embed(["고양이가 소파에서 잔다", "강아지가 공원에서 뛴다", "오늘 코스피가 올랐다"])
print("유사도 표:\n", np.round(v @ v.T, 2))

## 2. 자료 — 조각으로 나누기

In [ ]:
DOC = """
공방 이용 안내

공방은 평일 오전 9시부터 오후 6시까지 연다. 토요일은 오후 1시까지이고 일요일과 공휴일은 쉰다.

3D 프린터는 하루 최대 3시간까지 예약할 수 있다. 예약은 공방 입구의 태블릿에서 하고, 10분 넘게 늦으면 예약이 취소된다.

레이저 커터는 안전 교육을 들은 사람만 쓸 수 있다. 안전 교육은 매주 화요일 오후 4시에 30분 동안 한다.

납땜은 환기 장치를 켠 작업대에서만 한다. 납땜 인두는 쓰고 나서 반드시 거치대에 꽂고 전원을 끈다.

재료는 기본 필라멘트(PLA)와 합판 3mm 를 무료로 쓸 수 있다. 다른 재료는 가져와서 쓴다.
"""

def split(text, size=200):
    """빈 줄(문단)로 나누고, 긴 문단은 size 글자 안쪽으로 자른다."""
    out = []
    for para in [p.strip() for p in text.split("\n\n") if p.strip()]:
        while len(para) > size:
            cut = para.rfind(" ", 0, size)
            cut = cut if cut > 0 else size
            out.append(para[:cut]); para = para[cut:].strip()
        out.append(para)
    return out

chunks = split(DOC)
V = embed(chunks, "passage")
for i, c in enumerate(chunks):
    print(i, c[:40], "...")

## 3. 질문과 가까운 조각 찾기

In [ ]:
def search(q, k=2):
    s = V @ embed([q], "query")[0]                    # 길이가 1이라 내적 = 코사인 유사도
    order = np.argsort(-s)[:k]
    return [(float(s[i]), chunks[i]) for i in order]

for q in ["토요일에도 열어요?", "레이저 커터 쓰려면 어떻게 해요?", "주차장 있어요?"]:
    print("질문:", q)
    for score, c in search(q):
        print(f"   {score:.3f}  {c[:50]}")

## 4. 찾은 조각으로 답하기 — LLM
VLM(Gemma3 4B)을 글만 넣어 쓴다. **3.5GB** 라 처음 올리는 데 시간이 걸린다 (edge-lab 이 켜져 있으면 끄고 한다).
점수가 너무 낮으면(자료에 없는 질문) LLM 을 부르지 않는다 — 지어내지 않게.

In [ ]:
import openvino_genai as og

VLM_DIR = MODELS / "vlm" / "gemma3-4b-int4"
DEVICE = "GPU" if any(d.startswith("GPU") for d in core.available_devices) else "CPU"
t0 = time.perf_counter()
llm = og.VLMPipeline(str(VLM_DIR), DEVICE)
print(f"LLM 올림: {time.perf_counter() - t0:.0f}초 ({DEVICE})")

cfg = og.GenerationConfig()
cfg.max_new_tokens = 160
cfg.do_sample = False
cfg.repetition_penalty = 1.15

In [ ]:
MIN_SCORE = 0.60        # 이보다 낮으면 자료에 없는 질문으로 본다 (e5-small INT8 한국어 기준으로 맞춘 값)

def answer(q):
    hits = search(q, 2)
    if hits[0][0] < MIN_SCORE:
        return "자료에서 찾지 못했어요.", hits
    context = "\n".join(c for _, c in hits)
    prompt = (f"아래 자료만 보고 질문에 한국어로 짧게 답하세요.\n--- 자료 ---\n{context}\n--- 끝 ---\n"
              f"질문: {q}\n답:")
    return str(llm.generate(prompt, generation_config=cfg)).strip(), hits

for q in ["3D 프린터는 몇 시간까지 쓸 수 있어요?", "안전 교육은 언제 해요?", "주차장 있어요?"]:
    t0 = time.perf_counter()
    a, hits = answer(q)
    print(f"Q: {q}\nA: {a}\n   (근거 점수 {hits[0][0]:.2f} · {time.perf_counter() - t0:.1f}초)\n")

## 해 볼 것
1. `MIN_SCORE` 를 0.4 · 0.8 로 바꿔 보자. 무엇이 좋아지고 무엇이 나빠지나?
2. 조각 크기(`split(DOC, size=...)`)를 80 · 400 으로 바꾸면 찾기가 어떻게 달라지나?
3. 내 문서(.txt)를 읽어 `DOC` 대신 넣어 보자: `DOC = open("내문서.txt", encoding="utf-8").read()`
4. 찾기 없이 질문만 LLM 에 넣으면 어떻게 답하나? (지어내는지 보자)